Memanggil Data

In [3]:
import pandas as pd

# Buka file dataset
df = pd.read_csv('smart_lighting_dataset_2024.csv')

# Tampilkan 5 baris pertama
display(df.head())

,timestamp,zone_id,ambient_light_lux,motion_detected,temperature_celsius,occupancy_count,day_of_week,time_of_day,weather_condition,special_event_flag,energy_price_per_kwh,prev_hour_energy_usage_kwh,traffic_density,avg_pedestrian_speed,adjusted_light_intensity,energy_consumption_kwh,lighting_action_class
0,2024-01-01 00:00:00,3,0.00,0,22.08,7,Monday,Night,Cloudy,0,3.83,3.27,26.0,4.54,54.40,1.144,0
1,2024-01-01 00:43:00,3,35.78,0,19.19,6,Monday,Night,Cloudy,0,4.61,1.50,18.9,2.42,89.75,2.069,1
2,2024-01-01 01:27:00,7,0.00,0,27.78,9,Monday,Night,Clear,0,4.21,2.51,27.6,2.85,82.25,1.687,1
3,2024-01-01 02:11:00,2,0.00,1,33.11,16,Monday,Night,Foggy,0,6.16,0.34,23.1,2.45,72.57,1.631,1
4,2024-01-01 02:55:00,4,0.00,1,27.07,11,Monday,Night,Clear,0,5.72,1.52,26.4,3.49,78.79,1.737,1


Membuat "Kunci Jawaban" (Target)

In [4]:
def tentukan_kecerahan_lampu(row):
    if row['ambient_light_lux'] > 500: 
        return 0
    elif row['ambient_light_lux'] <= 500 and row['motion_detected'] == 1:
        return 100
    else:
        return 30

# Buat kolom baru berisi jawaban kecerahan lampu
df['light_intensity_percent'] = df.apply(tentukan_kecerahan_lampu, axis=1)

print("Target kecerahan berhasil dibuat!")
display(df[['ambient_light_lux', 'motion_detected', 'light_intensity_percent']].head())

Target kecerahan berhasil dibuat!


,ambient_light_lux,motion_detected,light_intensity_percent
0,0.00,0,30
1,35.78,0,30
2,0.00,0,30
3,0.00,1,100
4,0.00,1,100


Mengubah Teks Menjadi Angka (Persiapan ML)

In [7]:
from sklearn.model_selection import train_test_split

# Buang kolom yang tidak berguna untuk dipelajari
df_bersih = df.drop(columns=['timestamp', 'zone_id'])

# Ubah teks (hari, waktu, cuaca) menjadi angka 
kolom_kategori = ['day_of_week', 'time_of_day', 'weather_condition']
df_bersih = pd.get_dummies(df_bersih, columns=kolom_kategori)

# Pisahkan Soal (X) dan Kunci Jawaban (y)
X = df_bersih.drop(columns=['light_intensity_percent'])
y = df_bersih['light_intensity_percent']

# Potong data: 80% untuk latihan, 20% untuk ujian
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data sukses diubah ke angka dan dipotong!")

Data sukses diubah ke angka dan dipotong!


Melatih "Otak" AI (Random Forest) & Menyimpan Model

In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import joblib

# Panggil algoritma Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)

# Mulai proses belajar (Training)
print("AI sedang belajar, tunggu sebentar...")
model.fit(X_train, y_train)

# Tes kepintaran AI dengan data ujian
y_pred = model.predict(X_test)
error = mean_squared_error(y_test, y_pred)
print(f"Pelatihan Selesai! Tingkat Error: {error:.4f}")

# Simpan "Otak" AI-nya ke dalam file
joblib.dump(model, 'model_cerdas.pkl')
joblib.dump(X.columns.tolist(), 'fitur_model.pkl')
print("File 'model_cerdas.pkl' berhasil disimpan!")

AI sedang belajar, tunggu sebentar...
Pelatihan Selesai! Tingkat Error: 0.0001
File 'model_cerdas.pkl' berhasil disimpan!
